# Run pycisTopic on Distal regions/peaks (Step 1)

Build a cisTopic object using **only Distal peaks** (`peakType == "Distal"` in `FL_all_peaks.bed`), then export the binary matrix + names for Mallet topic modeling.

## Imports

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.sparse import csr_matrix
from scipy.io import mmwrite

import warnings
warnings.simplefilter(action="ignore")
import pycisTopic
print("pycisTopic version:", pycisTopic.__version__)
from pycisTopic.cistopic_class import create_cistopic_object


Matplotlib is building the font cache; this may take a moment.


pycisTopic version: 2.0a0


## CONFIG — edit to match your environment

In [2]:
WORK_DIR = "~/"
RES_DIR  = os.path.join(WORK_DIR, "results")
RUN_NAME = "01.pycisTopic_distal_UMAP"

# peak x cell matrix folder
MTX_DIR        = "~/PeakMatrix/"
COUNTS_MTX     = os.path.join(MTX_DIR, "counts.mtx")      # rows=peak, cols=cell
CELLNAMES_CSV  = os.path.join(MTX_DIR, "cellNames_fixed.csv")   # column "x"
PEAKNAMES_CSV  = os.path.join(MTX_DIR, "peakNames.csv")   # column "x"

# ArchR peak annotation BED (tab-separated, with header):
#   #chr left right names scores strands distToGeneStart nearestGene
#   peakType distToTSS nearestTSS GC
# The `names` column (e.g. "chr1:816084-816585") IS the peak id and matches peakNames.csv.
PEAK_BED         = os.path.join(WORK_DIR, "data", "FL_all_peaks.bed")
BED_NAMES_COL    = "names"      # peak id column == peakNames.csv "x"
BED_PEAKTYPE_COL = "peakType"   # Promoter / Exonic / Distal / Intronic
DISTAL_LABEL     = "Distal"

# cell metadata
CELL_META_CSV  = os.path.expanduser("/work/DevM_analysis/01.annotation/10.integration_joint_clean/data/FL_wnn_cellmeta.v02.csv")
CELLTYPE_COL   = "anno_wnn_v51"     # source column -> renamed to 'celltype'

# variability pre-filter (30th-percentile mean/variance cut)
APPLY_VARIABILITY_FILTER = True
PCTL = 30

Mallet_DIR = os.path.join(RES_DIR, RUN_NAME, "Mallet_res")
TMP_DIR    = os.path.join(Mallet_DIR, "tmp")
os.makedirs(TMP_DIR, exist_ok=True)
os.chdir(WORK_DIR)


## 1. Load peak x cell matrix

In [3]:
count_matrix = sc.read_mtx(COUNTS_MTX)          # rows x cols of the mtx
cellNames = pd.read_csv(CELLNAMES_CSV)["x"]
peakNames = pd.read_csv(PEAKNAMES_CSV)["x"]

df_sparse = csr_matrix(count_matrix.X)          # peak x cell
print(f"matrix shape (peak x cell): {df_sparse.shape}")
assert df_sparse.shape[0] == len(peakNames), "peakNames length != matrix rows"
assert df_sparse.shape[1] == len(cellNames), "cellNames length != matrix cols"


matrix shape (peak x cell): (575177, 308021)


## 2. Select Distal peaks from BED and subset matrix

In [8]:
bed = pd.read_csv(PEAK_BED, sep="\t")
# ---- 0-based -> 1-based ----
chr_col   = '#chr'
start_col = 'left'
end_col   = 'right'
bed[start_col] = bed[start_col].astype(int) + 1
bed[BED_NAMES_COL] = (
    bed[chr_col].astype(str)
    + ":" +
    bed[start_col].astype(str)
    + "-" +
    bed[end_col].astype(str)
)
# ---- 0-based -> 1-based ----
assert BED_PEAKTYPE_COL in bed.columns, f"'{BED_PEAKTYPE_COL}' not in {list(bed.columns)}"
assert BED_NAMES_COL in bed.columns, f"'{BED_NAMES_COL}' not in {list(bed.columns)}"

print("peakType value counts:")
print(bed[BED_PEAKTYPE_COL].value_counts().to_string())

distal_bed = bed[bed[BED_PEAKTYPE_COL].astype(str) == DISTAL_LABEL].copy()
print(f"distal peaks in BED: {len(distal_bed)} / {len(bed)}")
distal_ids = set(distal_bed[BED_NAMES_COL].astype(str))

is_distal = peakNames.isin(distal_ids).values
n_matched = int(is_distal.sum())
print(f"peakNames matched as Distal: {n_matched}")
if n_matched == 0:
    raise SystemExit(
        "No peakNames matched the Distal BED ids.\n"
        f"  peakNames[:3] = {list(peakNames[:3])}\n"
        f"  BED names[:3] = {list(distal_bed[BED_NAMES_COL][:3])}\n"
        "Adjust BED_NAMES_COL if id formats differ."
    )

df_distal   = df_sparse[is_distal, :]
peak_distal = pd.Series(peakNames[is_distal].values).reset_index(drop=True)


peakType value counts:
Intronic    299693
Distal      193700
Exonic       41172
Promoter     40612
distal peaks in BED: 193700 / 575177
peakNames matched as Distal: 193700


## 3. (optional) Variability filter on distal peaks

In [9]:
# if APPLY_VARIABILITY_FILTER:
#     means     = np.array(df_distal.mean(axis=1)).flatten()
#     variances = np.array(df_distal.power(2).mean(axis=1)).flatten() - means**2
#     mean_thr  = np.percentile(means, PCTL)
#     var_thr   = np.percentile(variances, PCTL)
#     keep_idx  = np.where((means > mean_thr) & (variances > var_thr))[0]
#     df_distal   = df_distal[keep_idx, :]
#     peak_distal = pd.Series(peak_distal.iloc[keep_idx].values).reset_index(drop=True)
#     print(f"peaks after variability filter: {len(peak_distal)}")

from scipy import sparse
import numpy as np
import pandas as pd

if APPLY_VARIABILITY_FILTER:
    n_cells = df_distal.shape[1]

    nnz_per_peak = df_distal.getnnz(axis=1)
    
    frac_per_peak = nnz_per_peak / float(n_cells)

    min_frac = 0.05
    keep_idx = np.where(frac_per_peak >= min_frac)[0]

    df_distal   = df_distal[keep_idx, :]
    peak_distal = pd.Series(peak_distal.iloc[keep_idx].values).reset_index(drop=True)

    print(f"peaks with signal in >= {min_frac*100:.1f}% cells: {len(peak_distal)}")

print(f"Final distal matrix (peak x cell): {df_distal.shape}")


peaks with signal in >= 5.0% cells: 13832
Final distal matrix (peak x cell): (13832, 308021)


## 4. Create cisTopic object + add cell metadata

In [10]:
cistopic_obj = create_cistopic_object(
    fragment_matrix=df_distal,
    cell_names=cellNames,
    region_names=peak_distal,
    tag_cells=False,
)

cell_data = pd.read_csv(CELL_META_CSV, index_col=0)
cell_data.index = cell_data.index.str.replace("_", "#")
cistopic_obj.add_cell_data(cell_data)
cistopic_obj.cell_data["celltype"] = cistopic_obj.cell_data[CELLTYPE_COL]
print(cistopic_obj)

2026-06-09 14:56:00,581 cisTopic     INFO     Creating CistopicObject
2026-06-09 14:56:05,375 cisTopic     INFO     Done!
CistopicObject from project cisTopic with n_cells × n_regions = 308021 × 13832


## 5. Export for Mallet + save object

In [12]:
binary_matrix = cistopic_obj.binary_matrix     # region x cell, binarized
mmwrite(os.path.join(TMP_DIR, "binary_accessibility_matrix.mtx"), binary_matrix)

pd.Series(cistopic_obj.cell_names).to_csv(
    os.path.join(TMP_DIR, "cellNames.txt"), index=False, header=False)
pd.Series(cistopic_obj.region_names).to_csv(
    os.path.join(TMP_DIR, "peakNames.txt"), index=False, header=False)

with open(os.path.join(RES_DIR, RUN_NAME, "cistopic_obj.pkl"), "wb") as fh:
    pickle.dump(cistopic_obj, fh)

print("Wrote binary matrix + cellNames.txt + peakNames.txt to", TMP_DIR)
print("Next: 02.MalletRun_createCorpus.sh -> 03.MalletRun_topic.sh -> 04.Models_assemble.sh")


Wrote binary matrix + cellNames.txt + peakNames.txt to ~/Mallet_res/tmp
Next: 02.MalletRun_createCorpus.sh -> 03.MalletRun_topic.sh -> 04.Models_assemble.sh
